# 🎬 RL Movie - ML-Agents トレーニング (Google Colab)

Unity ML-Agents の強化学習をクラウドで実行するためのノートブック。

## ワークフロー
1. Unity でビルド (.x86_64) を作成
2. Google Drive にアップロード
3. このノートブックでトレーニング実行
4. 学習済みモデル (.onnx) をダウンロード
5. Unity に戻してエージェントに適用

## 1. 環境セットアップ

In [ ]:
# ML-Agents Python パッケージのインストール
!pip install mlagents==1.1.0
!pip install protobuf==3.20.3  # 互換性のため

# バージョン確認
!mlagents-learn --help | head -5
print('✅ ML-Agents installed successfully!')

In [ ]:
# Google Drive マウント
from google.colab import drive
drive.mount('/content/drive')

# プロジェクトのパスを設定
PROJECT_DIR = '/content/drive/MyDrive/RL-Movie'
BUILD_DIR = f'{PROJECT_DIR}/Builds'
CONFIG_DIR = f'{PROJECT_DIR}/Configs'
RESULTS_DIR = f'{PROJECT_DIR}/Results'

import os
os.makedirs(BUILD_DIR, exist_ok=True)
os.makedirs(CONFIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'📁 Project dir: {PROJECT_DIR}')
print(f'📁 Build dir:   {BUILD_DIR}')
print(f'📁 Config dir:  {CONFIG_DIR}')
print(f'📁 Results dir: {RESULTS_DIR}')

## 2. ビルドの準備

Unity で Linux ビルドを作成し、Google Drive の `RL-Movie/Builds/` にアップロードしてください。

### Unity で Linux ビルドを作成する手順:
1. **File > Build Settings**
2. Platform を **Linux** に変更（Linux Build Support モジュールが必要）
3. **Player Settings** で:
   - Resolution and Presentation > Fullscreen Mode: **Windowed**
   - Other Settings > Scripting Backend: **Mono** (推奨)
4. **Build** をクリック
5. ビルド成果物を Google Drive にアップロード

In [ ]:
# ビルドファイルの確認
import glob

builds = glob.glob(f'{BUILD_DIR}/*.x86_64')
if builds:
    print('🎮 Available builds:')
    for b in builds:
        print(f'  - {os.path.basename(b)}')
else:
    print('⚠️ No builds found. Upload a Linux build to Google Drive first.')

In [ ]:
# ビルドに実行権限を付与
# BUILD_NAME を実際のビルドファイル名に変更してください
BUILD_NAME = 'RLMovie.x86_64'  # ← ここを変更
BUILD_PATH = f'{BUILD_DIR}/{BUILD_NAME}'

!chmod +x {BUILD_PATH}
print(f'✅ Executable permissions set for {BUILD_NAME}')

## 3. トレーニング設定

YAML 設定ファイルを Google Drive の `RL-Movie/Configs/` に配置するか、
以下のセルで直接作成します。

In [ ]:
# トレーニング設定をファイルに書き出し
CONFIG_FILE = f'{CONFIG_DIR}/training_config.yaml'

config = """
behaviors:
  TemplateAgent:
    trainer_type: ppo
    hyperparameters:
      batch_size: 128
      buffer_size: 2048
      learning_rate: 3.0e-4
      beta: 5.0e-3
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
      learning_rate_schedule: linear
    network_settings:
      normalize: true
      hidden_units: 128
      num_layers: 2
    reward_signals:
      extrinsic:
        gamma: 0.99
        strength: 1.0
    max_steps: 500000
    time_horizon: 64
    summary_freq: 10000
    keep_checkpoints: 5
    checkpoint_interval: 50000
"""

with open(CONFIG_FILE, 'w') as f:
    f.write(config)

print(f'✅ Config saved to {CONFIG_FILE}')

## 4. トレーニング実行

In [ ]:
# トレーニングの実行
RUN_ID = 'run_001'  # ← 実行ごとに変更

!mlagents-learn {CONFIG_FILE} \
    --env={BUILD_PATH} \
    --run-id={RUN_ID} \
    --no-graphics \
    --results-dir={RESULTS_DIR} \
    --force

## 5. TensorBoard でモニタリング

In [ ]:
# TensorBoard の起動
%load_ext tensorboard
%tensorboard --logdir {RESULTS_DIR}

## 6. 学習済みモデルのダウンロード

In [ ]:
# 学習済みモデルファイルの確認
import glob

models = glob.glob(f'{RESULTS_DIR}/{RUN_ID}/**/*.onnx', recursive=True)
if models:
    print('🧠 Trained models:')
    for m in models:
        print(f'  - {m}')
    
    # 最新のモデルをダウンロード用にコピー
    latest_model = models[-1]
    download_path = f'{PROJECT_DIR}/{RUN_ID}_model.onnx'
    !cp {latest_model} {download_path}
    print(f'\n📥 Model copied to: {download_path}')
    print('\n→ Google Drive からダウンロードして Unity の')
    print('  Assets/StreamingAssets/ に配置してください。')
else:
    print('⚠️ No models found. Training may not have completed.')

In [ ]:
# 直接ダウンロード (Colab から)
from google.colab import files

if models:
    files.download(models[-1])
    print('✅ Download started!')